# 건강검진 코호트 EDA (10만 명 · 5년 · 분기별)

**이 노트북은 EDA만 한다. 모델 학습 없음.**

목적: 어떤 지표가 실제로 예측력이 있는지 확인해서, 나중에 모델에 넣을 특성을 정한다.

핵심 주의점
- `grade`(종합판정)는 검진 지표들을 합산해 만든 값이다. 그래서 **`grade`와의 상관은 발견이 아니라 정의상 종속**이다.
- 특성 선택은 실제로 예측하려는 타겟 —
  **"현재 정상·주의인 사람이 다음 분기에 위험으로 전환되는가"** — 기준으로 판단한다. (섹션 6)

## 0. 환경 준비

In [ ]:
!pip -q install pyarrow
!apt-get install -y fonts-nanum > /dev/null 2>&1

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, warnings
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font='NanumGothic')
pd.set_option('display.width', 200)
print('준비 완료')

## 1. 데이터 로드

In [ ]:
import os

PATH = '/content/checkups.parquet'   # 코랩 왼쪽 [파일] 탭에 드래그해서 올린 위치

assert os.path.exists(PATH), (
    '파일이 없음. 코랩 왼쪽 [파일] 탭(폴더 아이콘)에 checkups.parquet 을 드래그해서 '
    '업로드가 끝난 뒤 이 셀을 다시 실행하세요.')
print(f'파일 확인 {os.path.getsize(PATH)/1e6:.0f}MB')

df = pd.read_parquet(PATH)
print(f'{len(df):,} 행 · {df.person_id.nunique():,} 명 · {df.quarter.nunique()} 분기')

# 영문 컬럼 -> 한글 이름 (그래프/표에 같이 표기)
KOR = {
    'bmi':'체질량지수(BMI)', 'waist':'허리둘레', 'systolic':'수축기혈압',
    'diastolic':'이완기혈압', 'fbs':'공복혈당', 'total_chol':'총콜레스테롤',
    'triglyceride':'중성지방', 'hdl':'HDL(좋은콜)', 'ldl':'LDL(나쁜콜)',
    'hemoglobin':'혈색소', 'ast':'간수치AST', 'alt':'간수치ALT',
    'ggt':'감마지티피', 'creatinine':'크레아티닌',
    'age':'나이', 'grade':'종합판정', 'transition':'위험전환',
}
kn = lambda c: KOR.get(c, c)

METRICS = ['bmi','waist','systolic','diastolic','fbs','total_chol',
           'triglyceride','hdl','ldl','hemoglobin','ast','alt','ggt','creatinine']
df.head()

## 2. 기본 정보 / 결측 / 수검 횟수

In [ ]:
print('=== 형태 ===');  print(df.shape)
print('\n=== 타입 ===');  print(df.dtypes)
print('\n=== 결측 ===');  print(df.isna().sum()[lambda s: s>0] if df.isna().any().any() else '없음 (미수검은 행 자체가 빠짐)')

print('\n=== 인원별 관측 수 ===')
cnt = df.groupby('person_id').size()
print(cnt.describe().round(2))
print(f'\n20회 미만(미수검 있음) 인원 비율: {(cnt<20).mean()*100:.1f}%')

df.describe().T.round(2)

## 3. 인구 분포 (연령·성별·흡연·판정)

In [ ]:
base = df[df.quarter == 0].copy()
def band(a):
    return '20대이하' if a<30 else '30대' if a<40 else '40대' if a<50 else '50대' if a<60 else '60대' if a<70 else '70대+'
base['연령대'] = base.age.map(band)
order = ['20대이하','30대','40대','50대','60대','70대+']

fig, ax = plt.subplots(2, 2, figsize=(14, 9))
sns.countplot(data=base, x='연령대', hue='sex', order=order, ax=ax[0,0]); ax[0,0].set_title('연령대×성별 분포')
sns.countplot(data=df, x='grade', hue='sex', ax=ax[0,1]); ax[0,1].set_title('종합판정×성별 (0정상 1주의 2위험)')
sns.countplot(data=base, x='smoker', hue='sex', ax=ax[1,0]); ax[1,0].set_title('흡연×성별')
sns.histplot(data=base, x='age', hue='sex', bins=30, ax=ax[1,1]); ax[1,1].set_title('연령 분포')
plt.tight_layout(); plt.show()

print('판정 비율:', (df.grade.value_counts(normalize=True).sort_index()*100).round(1).to_dict(), '(목표 40.2 / 32.2 / 27.6)')
print('흡연율: 전체 %.1f%% / 남 %.1f%% / 여 %.1f%%' % (
    base.smoker.mean()*100, base[base.sex=='M'].smoker.mean()*100, base[base.sex=='F'].smoker.mean()*100))

## 4. 검진 지표 분포 & 이상치

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 12))
for ax, c in zip(axes.ravel(), METRICS):
    sns.histplot(df[c], bins=60, ax=ax)
    ax.set_title(f'{kn(c)}  [{c}]', fontsize=10); ax.set_xlabel('')
for ax in axes.ravel()[len(METRICS):]:
    ax.axis('off')
plt.tight_layout(); plt.show()

print('=== 이상치 비율 (IQR 1.5배 기준) ===')
print(f"  {'지표':<16}{'비율':>7}{'최소':>9}{'최대':>11}")
for c in METRICS:
    q1, q3 = df[c].quantile([.25, .75]); iqr = q3 - q1
    out = ((df[c] < q1-1.5*iqr) | (df[c] > q3+1.5*iqr)).mean()*100
    print(f'  {kn(c):<14} {out:5.2f}%  {df[c].min():8.1f}{df[c].max():11.1f}')

print()
print('해석 1) 중성지방·감마지티피·AST·ALT는 로그정규 분포라 IQR 기준 이상치가 원래 많이 잡힘')
print('        -> 실제 오류가 아님. 모델링 단계에서 로그변환 대상.')
print('해석 2) 혈압 500대, 혈당 500대 같은 값은 생리학적으로 불가능 -> 주입한 측정오류.')
print('        -> 모델링 단계에서 클리핑 대상.')

## 5. 상관관계 (지표끼리 / 겹치는 정보 확인)

In [ ]:
corr = df[METRICS + ['age','grade']].corr()
lab = [kn(c) for c in corr.columns]
plt.figure(figsize=(13, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True,
            xticklabels=lab, yticklabels=lab, cbar_kws={'shrink':.8})
plt.title('지표 상관관계'); plt.tight_layout(); plt.show()

c_grade = corr['grade'].drop('grade').abs().sort_values(ascending=False)

print('=== 겹치는 정보 (지표끼리 |상관| >= 0.7) ===')
cm = corr.drop(index='grade', columns='grade')
for a in cm.columns:
    for b in cm.columns:
        if a < b and abs(cm.loc[a, b]) >= 0.7:
            print(f'  {kn(a)} <-> {kn(b)}   {cm.loc[a,b]:+.2f}   거의 같은 정보')

print()
print('!! 주의: grade(종합판정)는 지표들을 합산해 만든 값이다.')
print('   따라서 grade와의 상관은 "발견"이 아니라 정의상 종속이다.')
print('   -> 특성 선택 기준으로 쓰면 안 된다. 섹션 6에서 실제 타겟 기준으로 다시 본다.')

## 6. ★ 실제 예측 타겟(위험 전환)과의 상관 — 특성 선택은 여기서 결정

**타겟 정의**: 현재 *정상·주의* 인 사람이 → **다음 분기에 위험으로 전환**되는가 (0/1)

- 이미 위험인 사람은 제외한다. 이미 아픈 사람을 맞히는 건 임상 가치가 없다.
- 전이 관성이 강해서(위험→위험 75.5%) 그냥 "다음 분기 위험 여부"를 맞히면
  모델이 **"지금 나쁘면 다음에도 나쁘다"** 만 학습한다. 그건 의사가 이미 안다.
- 위험군을 빼면 모델은 **변화·추세**를 봐야만 맞힐 수 있게 된다.

In [ ]:
# t+1 분기 판정을 t 행에 붙임 (실제로 다음 분기 검진이 있는 행만 남음)
nxt = df[['person_id','quarter','grade']].rename(columns={'grade':'next_grade'}).copy()
nxt['quarter'] -= 1
e = df.merge(nxt, on=['person_id','quarter'], how='inner')

elig = e[e.grade < 2].copy()                       # 현재 정상·주의만
elig['transition'] = (elig.next_grade == 2).astype(int)

print(f'전체 {len(df):,}행 -> 다음분기 검진 있는 행 {len(e):,} -> 현재 정상·주의 {len(elig):,}')
print(f'양성률(위험 전환)  {elig.transition.mean()*100:.2f}%')
print()
print('현재 판정별 전환율:')
for gv, nm in [(0,'정상'), (1,'주의')]:
    sub = elig[elig.grade == gv]
    print(f'  {nm}({gv})  {len(sub):>9,}행  ->  위험 전환 {sub.transition.mean()*100:5.2f}%')

In [ ]:
# ---- 두 타겟과의 상관 비교 ----
c_trans = elig[METRICS + ['age','transition']].corr()['transition'].drop('transition')

cmp = pd.DataFrame({
    'grade상관(정의상·참고)': c_grade.reindex(c_trans.index).round(3),
    '전환상관(실제타겟)':      c_trans.abs().round(3),
    '방향': np.where(c_trans >= 0, '↑높을수록위험', '↓낮을수록위험'),
})
cmp['거품비율'] = (cmp['전환상관(실제타겟)'] / cmp['grade상관(정의상·참고)']).round(2)
cmp = cmp.sort_values('전환상관(실제타겟)', ascending=False)
cmp.index = [kn(c) for c in cmp.index]

print('=== 특성별 예측력 (전환상관 내림차순) ===')
print(cmp.to_string())
print()
print('거품비율 = 전환상관 / grade상관.  낮을수록 grade 계산식에 직접 들어간 지표라')
print('grade 상관이 부풀려져 있었다는 뜻이다.')

In [ ]:
# ---- 시각화 + 제외 후보 ----
THRESH = 0.05
o = c_trans.abs().sort_values()
plt.figure(figsize=(9, 6))
plt.barh([kn(i) for i in o.index], o.values,
         color=['#c0392b' if v >= .22 else '#e67e22' if v >= .15 else
                '#f1c40f' if v >= THRESH else '#bdc3c7' for v in o.values])
plt.axvline(THRESH, ls='--', c='k', lw=1)
plt.xlabel('|상관|  (다음 분기 위험 전환)')
plt.title(f'실제 타겟과의 상관 — 회색 = |상관| < {THRESH} (제외 후보)')
plt.tight_layout(); plt.show()

WEAK = c_trans[c_trans.abs() < THRESH].index.tolist()
KEEP = [c for c in METRICS if c not in WEAK]
print(f'제외 후보 ({len(WEAK)}개): {[kn(c) for c in WEAK] or "없음"}')
print(f'유지     ({len(KEEP)}개): {[kn(c) for c in KEEP]}')

## 7. 시계열 추세 (5년) & 판정 전이 행렬

In [ ]:
q = df.groupby('quarter').agg(
    bmi=('bmi','mean'), systolic=('systolic','mean'), fbs=('fbs','mean'),
    ldl=('ldl','mean'), 위험비율=('grade', lambda s: (s==2).mean()*100)).reset_index()

fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
for c in ['bmi','systolic','fbs','ldl']:
    ax[0].plot(q.quarter, q[c]/q[c].iloc[0]*100, marker='o', label=kn(c))
ax[0].set_title('지표 평균 추이 (첫 분기 = 100)'); ax[0].set_xlabel('분기'); ax[0].legend()
ax[1].plot(q.quarter, q.위험비율, marker='o', color='crimson')
ax[1].set_title('위험 판정 비율(%) 추이'); ax[1].set_xlabel('분기')
plt.tight_layout(); plt.show()

print('5년(20분기) 변화율:')
for c in ['bmi','systolic','fbs','ldl']:
    print(f'  {kn(c):<14} {(q[c].iloc[-1]/q[c].iloc[0]-1)*100:+.2f}%')
print(f'  위험 판정 비율 {q.위험비율.iloc[0]:.1f}% -> {q.위험비율.iloc[-1]:.1f}%  ({q.위험비율.iloc[-1]-q.위험비율.iloc[0]:+.1f}%p)')
print()

# 판정 전이 행렬
d = df.sort_values(['person_id','quarter'])
d['ng'] = d.groupby('person_id')['grade'].shift(-1)
trans = pd.crosstab(d.grade, d.ng, normalize='index').round(3)
trans.index = ['정상','주의','위험']; trans.columns = ['정상','주의','위험']
print('=== 판정 전이확률 (행 = 현재, 열 = 다음 분기) ===')
print(trans.to_string())
print()
print(f'-> 대각선(상태 유지)이 {np.diag(trans).min():.3f} 이상. 관성이 강하다.')
print('   그래서 "다음 분기 위험 여부"를 그냥 예측하면 현재 상태 복사만 배운다.')
print('   -> 섹션 6처럼 위험군을 제외한 "전환" 예측이 맞다.')

plt.figure(figsize=(5.5, 4.5))
sns.heatmap(trans, annot=True, fmt='.3f', cmap='Blues', cbar=False)
plt.title('판정 전이확률'); plt.xlabel('다음 분기'); plt.ylabel('현재')
plt.tight_layout(); plt.show()

## 8. 연령대 · 성별 위험도

In [ ]:
df['연령대'] = df.age.map(band)
p = df.groupby(['연령대','sex'])['grade'].apply(lambda s: (s==2).mean()*100).unstack().round(1)
p = p.reindex(order)
print('=== 연령대×성별 위험 판정 비율(%) ===')
print(p.to_string())

t = elig.copy(); t['연령대'] = t.age.map(band)
pt = t.groupby(['연령대','sex'])['transition'].mean().unstack().mul(100).round(2).reindex(order)
print('\n=== 연령대×성별 위험 "전환" 비율(%) — 현재 정상·주의만 ===')
print(pt.to_string())

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
p.plot(kind='bar', ax=ax[0]); ax[0].set_title('위험 판정 비율(%)'); ax[0].set_ylabel('%')
pt.plot(kind='bar', ax=ax[1]); ax[1].set_title('위험 전환 비율(%) — 조기경보 대상'); ax[1].set_ylabel('%')
for a in ax: a.set_xlabel(''); a.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

## 9. 개인 궤적 샘플 (시계열성 확인)

In [ ]:
ids = df.person_id.drop_duplicates().sample(6, random_state=0).tolist()
fig, axes = plt.subplots(2, 3, figsize=(16, 7))
for ax, pid in zip(axes.ravel(), ids):
    g = df[df.person_id==pid].sort_values('quarter')
    ax.plot(g.quarter, g.systolic, marker='o', label='수축기혈압')
    ax.plot(g.quarter, g.fbs, marker='s', label='공복혈당')
    ax2 = ax.twinx(); ax2.step(g.quarter, g.grade, color='gray', alpha=.5, where='mid')
    ax2.set_ylim(-0.2, 2.2); ax2.set_yticks([0,1,2]); ax2.set_yticklabels(['정상','주의','위험'])
    ax.set_title(f'person {pid} ({g.sex.iloc[0]}, {g.age.iloc[0]:.0f}세)')
    ax.set_xlabel('분기'); ax.legend(fontsize=8, loc='upper left')
plt.tight_layout(); plt.show()
print('회색 계단 = 종합판정. 개인별로 오르내리는 시계열성이 보이면 정상.')

## 10. EDA 요약

In [ ]:
print('='*68)
print('EDA 요약')
print('='*68)
print(f'1) 규모      {len(df):,}행 · {df.person_id.nunique():,}명 · {df.quarter.nunique()}분기(5년)')
print(f'   판정 분포 {(df.grade.value_counts(normalize=True).sort_index()*100).round(1).tolist()} (정상/주의/위험)')
print(f'   결측      셀 결측 없음. 미수검은 행 자체가 빠짐 (20회 미만 {(cnt<20).mean()*100:.1f}%)')
print()
print(f'2) 예측 타겟  현재 정상·주의 -> 다음 분기 위험 전환')
print(f'   대상       {len(elig):,}행 · 양성률 {elig.transition.mean()*100:.2f}%')
print(f'   정상 {elig[elig.grade==0].transition.mean()*100:.2f}% / 주의 {elig[elig.grade==1].transition.mean()*100:.2f}%')
print()
print(f'3) 유지할 지표 ({len(KEEP)}개 + 나이)')
for c in c_trans.abs().sort_values(ascending=False).index:
    if c in KEEP or c == 'age':
        print(f'   {kn(c):<16} 상관 {abs(c_trans[c]):.3f}')
print(f'   제외: {[kn(c) for c in WEAK]}  (상관 {THRESH} 미만)')
print()
print('4) 모델링 단계에서 할 전처리')
print('   - 클리핑   혈압/혈당/중성지방/간수치 생리학적 상한 (주입된 측정오류 제거)')
print('   - 로그변환 중성지방, 감마지티피, AST, ALT (로그정규 분포)')
print('   - 파생특성 lag1(직전값) / d1(변화량) / ma4(1년평균) / std4(변동성) / dev(평균대비이탈)')
print('   - 분할     GroupShuffleSplit (사람 단위. 같은 사람이 train/test에 겹치면 누출)')
print('   - 제외     severity (데이터 생성기 내부 점수. 실제 병원에 없음)')
print('='*68)